[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/010049nn/EZ_pair_graph/blob/main/demo_ez_pair_graph.ipynb)

# EZ-Pair Graph Demo

**Scalable unified-axis visualization for summarizing large-scale paired data**

This notebook demonstrates EZ-Pair Graph using the included test datasets.

**Reference:** Ezoe, A., Seki, M. & Mochida, K. EZ-Pair Graph: Scalable unified-axis visualization for summarizing large-scale paired data. *Bioinformatics Advances* (2026).

**License:** Academic and non-commercial research use only (RIKEN).


> **Running on Google Colab:** Run the cell below first.
> If you already have `pip install` done locally, skip this cell.


In [ ]:
# === Google Colab Setup ===
!pip install git+https://github.com/010049nn/EZ_pair_graph.git -q

# Download test datasets
import urllib.request, os
base = "https://raw.githubusercontent.com/010049nn/EZ_pair_graph/main/"
for f in ["test_dataset.txt", "test_dataset_n1000.txt"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(base + f, f)
        print(f"Downloaded {f}")
print("Setup complete!")


## 1. Import


In [ ]:
import ez_pair_graph as ezpg
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

print(f"ez-pair-graph version: {ezpg.__version__}")


## 2. test_dataset.txt (n=120)

120 paired observations. Generate all four plot types at once.


In [ ]:
# Generate all 4 plots from test_dataset.txt
figures = ezpg.plot("test_dataset.txt", output_dir="output_test", format="png")

for name, path in figures.items():
    print(f"\n--- {name} ---")
    display(IPImage(filename=path, width=600))


## 3. test_dataset_n1000.txt (n=1000)

Larger dataset. Show cluster numbers and sample counts, hide outliers.


In [ ]:
# Large dataset (n=1000) with show-numbers and no-outliers
figures = ezpg.plot(
    "test_dataset_n1000.txt",
    output_dir="output_n1000",
    format="png",
    show_numbers=True,
    no_outliers=True
)

for name, path in figures.items():
    print(f"\n--- {name} ---")
    display(IPImage(filename=path, width=600))


## 4. Low-Level API — Individual Plots

Step through clustering, statistics, and plotting explicitly.


In [ ]:
from ez_pair_graph.preparation import cluster_data, compute_statistics, load_data
from ez_pair_graph.plotting import (
    plot_slopegraph, plot_trapezoid,
    plot_clustered_lines, plot_parallel_arrows
)

# Load data
x, y = load_data("test_dataset.txt")
print(f"Loaded {len(x)} pairs")

# Cluster
result = cluster_data(x, y, method="hierarchical", max_k=7)
clusters = result["clusters"]
print(f"Number of clusters: {result['n_clusters']}")

# Compute statistics
stats = compute_statistics(x, y, clusters)


In [ ]:
# Slope Graph
fig, ax = plot_slopegraph(x, y, alpha=0.3)
plt.show()


In [ ]:
# Trapezoid Plot
fig, ax = plot_trapezoid(x, y)
plt.show()


In [ ]:
# Clustered Line Plot
fig, ax = plot_clustered_lines(x, y, clusters, stats)
plt.show()


In [ ]:
# Parallel Arrow Plot
fig, ax = plot_parallel_arrows(x, y, clusters, stats)
plt.show()


## 5. Clustering Method Comparison

Compare Hierarchical (Ward) and HDBSCAN side by side.


In [ ]:
result_hier = cluster_data(x, y, method="hierarchical", max_k=7)
result_hdb  = cluster_data(x, y, method="hdbscan", min_cluster_size=3)

stats_hier = compute_statistics(x, y, result_hier["clusters"])
stats_hdb  = compute_statistics(x, y, result_hdb["clusters"])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_clustered_lines(x, y, result_hier["clusters"], stats_hier, ax=axes[0])
axes[0].set_title(f"Hierarchical ({result_hier['n_clusters']} clusters)")

plot_clustered_lines(x, y, result_hdb["clusters"], stats_hdb, ax=axes[1])
axes[1].set_title(f"HDBSCAN ({result_hdb['n_clusters']} clusters)")

fig.suptitle("Clustering Method Comparison", fontsize=14)
plt.tight_layout()
plt.show()


## 6. Log2 Transformation

Useful for gene expression data or other log-scale measurements.


In [ ]:
# Visualize test_dataset.txt with log2 transformation
figures = ezpg.plot(
    "test_dataset.txt",
    output_dir="output_log2",
    format="png",
    log2=True
)

for name, path in figures.items():
    print(f"\n--- {name} (log2) ---")
    display(IPImage(filename=path, width=600))


## 7. Output Options

Control output format (pdf/svg/png), filename prefix, and which plots to generate.


In [ ]:
# Generate specific plots with a filename prefix
figures = ezpg.plot(
    "test_dataset.txt",
    output_dir="output_options",
    format="png",
    output_prefix="experiment1",
    plots=["trapezoid", "clustered_line"],
    show_numbers=True
)

for name, path in figures.items():
    print(f"\n--- {name} ---")
    display(IPImage(filename=path, width=600))


## 8. Cluster Statistics

`compute_statistics()` returns detailed per-cluster summary statistics.


In [ ]:
result = cluster_data(x, y)
stats = compute_statistics(x, y, result["clusters"])

print(f"Total clusters: {len(stats)}\n")

for cid in sorted(stats.keys()):
    s = stats[cid]
    c = s["calculated"]
    direction = "ascending" if c["mean_point"][1] > c["mean_point"][0] else "descending"
    print(f'Cluster {cid}: n={c["n"]:3d}  '
          f'X={c["mean_point"][0]:6.2f} -> Y={c["mean_point"][1]:6.2f}  '
          f'({direction})')


## 9. Command-Line Interface (CLI)

```bash
# Basic usage
ez-pair-graph test_dataset.txt

# PNG output with cluster numbers and no outliers
ez-pair-graph test_dataset.txt --format png --show-numbers --no-outliers

# HDBSCAN clustering with log2 and filename prefix
ez-pair-graph test_dataset.txt --method hdbscan --min-cluster-size 3 --log2 --output-prefix exp1

# Generate specific plots only
ez-pair-graph test_dataset.txt --plots trapezoid clustered_line
```


In [ ]:
# Run CLI on Colab
!ez-pair-graph test_dataset.txt --format png --show-numbers --output-dir output_cli

import glob
for path in sorted(glob.glob("output_cli/*.png")):
    print(f"\n--- {os.path.basename(path)} ---")
    display(IPImage(filename=path, width=600))


## Summary

| Function | Input | Description |
|----------|-------|-------------|
| `ezpg.plot(file)` | File path | Generate all plots from file |
| `ezpg.plot_array(x, y)` | NumPy arrays | Generate all plots from arrays |
| `ezpg.plot_dataframe(df)` | DataFrame | Generate all plots from DataFrame |
| `plot_slopegraph(x, y)` | arrays | Slope graph (individual) |
| `plot_trapezoid(x, y)` | arrays | Trapezoid plot (individual) |
| `plot_clustered_lines(x, y, c, s)` | arrays + stats | Clustered line plot (individual) |
| `plot_parallel_arrows(x, y, c, s)` | arrays + stats | Parallel arrow plot (individual) |
| `cluster_data(x, y)` | arrays | Clustering |
| `compute_statistics(x, y, c)` | arrays + labels | Per-cluster statistics |
